# Connect to Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import libraries


In [2]:
!pip install -q datasets transformers rank-bm25 sentence-transformers tqdm evaluate torch accelerate
!pip install numpy --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 66.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.4 which is incompatible.
cupy-cuda12x 13.3.0 requires numpy<2.3,>=1.22, but you have numpy 2.3.4 which is incompa

# Utility functions


In [1]:
from datasets import load_dataset
import json
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import gc
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc

base_path = "/content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val"

def load_jsonl(path):
    data = []
    print(f"Attempting to load JSONL file: {path}") # Thêm log
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                try:
                    # Loại bỏ khoảng trắng thừa ở đầu/cuối dòng
                    processed_line = line.strip()
                    if processed_line: # Đảm bảo dòng không rỗng sau khi strip
                        data.append(json.loads(processed_line))
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON on line {i+1} in {path}: {e}")
                    print(f"Problematic line content (first 100 chars): {line[:100]}...")
                    # Tùy chọn: Bỏ qua dòng lỗi hoặc dừng hẳn
                    # raise e # Dừng hẳn nếu muốn biết lỗi ngay lập tức
                    print("Skipping this problematic line.") # Bỏ qua dòng lỗi và tiếp tục
    except FileNotFoundError:
        print(f"ERROR: File not found at {path}")
        raise
    except Exception as e:
        print(f"An unexpected error occurred while loading {path}: {e}")
        raise
    print(f"Successfully loaded {len(data)} records from {path}")
    return data

def load_qrels(path):
    qrels = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            qid, _, docid, rel = line.strip().split()
            qrels.setdefault(qid, {})[docid] = int(rel)
    return qrels

# Import dataset if not exists


In [2]:
def save_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for d in items:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

def save_qrels(qrels_list, path):
    # qrels_list là list của (query_id, doc_id, rel)
    with open(path, "w", encoding="utf-8") as f:
        # mỗi dòng: query_id \t 0 \t doc_id \t relevance
        for qid, docid, rel in qrels_list:
            f.write(f"{qid}\t0\t{docid}\t{rel}\n")

def convert_ms_marco(split="validation", out_dir=os.path.join(base_path, "data/ms_marco_val")):
    ds = load_dataset("microsoft/ms_marco", 'v2.1', split=split)
    os.makedirs(out_dir, exist_ok=True)

    # Queries
    queries = [{"id": str(rec["query_id"]), "text": rec["query"]} for rec in ds]

    # Corpus & Qrels
    corpus_map = {}
    qrels = []

    for rec in ds:
        qid = str(rec["query_id"])
        passages = rec["passages"]
        texts = passages["passage_text"]
        selected = passages["is_selected"]

        for idx, text in enumerate(texts):
            if not text:
                continue
            doc_id = f"{qid}_{idx}"
            corpus_map[doc_id] = text
            if selected[idx] == 1:
                qrels.append((qid, doc_id, 1))

    corpus = [{"id": did, "text": txt} for did, txt in corpus_map.items()]

    # Save
    save_jsonl(corpus, os.path.join(out_dir, "corpus.jsonl"))
    save_jsonl(queries, os.path.join(out_dir, "queries.jsonl"))
    save_qrels(qrels, os.path.join(out_dir, "qrels.tsv"))

    print("Done convert MS MARCO split:", split)

# if exists the folder, skip
if not os.path.exists(os.path.join(base_path, "corpus.jsonl")):
    convert_ms_marco(split="validation", out_dir=base_path)

# if not os.path.exists(os.path.join(base_path, "data/ms_marco_train/corpus.jsonl")):
#     convert_ms_marco(split="train", out_dir=base_path)


In [3]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from tqdm import tqdm
import math

class Reranker:
    def __init__(self, model_name, model_type="cross-encoder", device="cuda", batch_size=32):
        """
        Khởi tạo Reranker.
        model_type có thể là 'cross-encoder' hoặc 'qwen-reranker'.
        """
        if not torch.cuda.is_available() and device == "cuda":
            print(f"CUDA not available for {model_name}, switching to CPU. Reranking will be very slow.")
            device = "cpu"
        self.device = device
        self.model_name = model_name
        self.model_type = model_type.lower()
        self.batch_size = batch_size # Lưu batch_size vào instance

        print(f"Loading tokenizer for {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

        print(f"Loading model {model_name} ({self.model_type})...")
        if self.model_type == 'qwen-reranker':
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16, # Qwen hoạt động tốt với float16
                attn_implementation="sdpa", # Sử dụng SDPA (thay vì flash_attention_2) cho tương thích rộng hơn
                trust_remote_code=True
            ).to(self.device).eval()
            # Pre-compile tokens and templates for Qwen
            self._qwen_prepare()
        elif self.model_type == 'cross-encoder':
            self.model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                 # Sử dụng float16 nếu GPU hỗ trợ và model tương thích, nếu không dùng float32
                torch_dtype=torch.float16 if self.device == 'cuda' else torch.float32
            ).to(self.device).eval()
            # Cross-encoder thường có 1 output logit cho relevance, hoặc 2 (relevant/irrelevant)
            self.num_labels = self.model.config.num_labels
            print(f"Cross-encoder loaded with {self.num_labels} output labels.")
        else:
            raise ValueError("Unsupported model_type. Choose 'cross-encoder' or 'qwen-reranker'.")

        print(f"Reranker ({model_name}) initialized successfully on {self.device}.")

    def _qwen_prepare(self):
        """Helper to prepare Qwen specific tokens and templates."""
        self.token_false_id = self.tokenizer.convert_tokens_to_ids("no")
        self.token_true_id = self.tokenizer.convert_tokens_to_ids("yes")
        self.prefix = ("<|im_start|>system\n"
                       "Judge whether the Document meets the requirements based on the Query and the Instruct provided. "
                       "Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n"
                       "<|im_start|>user\n")
        self.suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
        self.prefix_tokens = self.tokenizer.encode(self.prefix, add_special_tokens=False)
        self.suffix_tokens = self.tokenizer.encode(self.suffix, add_special_tokens=False)
        # Qwen3 Reranker hỗ trợ context dài hơn, nhưng rerank thường dùng context ngắn hơn
        # Lấy max_length từ config nếu có, nếu không đặt giá trị an toàn
        self.max_length = getattr(self.model.config, 'max_position_embeddings', 8192)
        print(f"Qwen reranker max sequence length: {self.max_length}")


    def _format_qwen_input(self, instruction, query, doc):
        return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}"

    @torch.no_grad()
    def rerank(self, query, docs, k=10):
        """
        Reranks a list of documents for a given query using the specific model type.
        """
        if not docs: return []

        # Chuẩn bị input dựa trên model type
        doc_map = {} # Dùng để lưu trữ thông tin gốc của doc
        input_pairs = [] # List các input cho model (text pairs hoặc formatted text)

        for idx, doc in enumerate(docs):
            doc_id = doc.get("doc_id")
            doc_text = doc.get("text", "")
            if not doc_id or not doc_text: continue

            if self.model_type == 'qwen-reranker':
                 instruction = 'Given a web search query, retrieve relevant passages that answer the query'
                 input_pairs.append(self._format_qwen_input(instruction, query, doc_text))
            elif self.model_type == 'cross-encoder':
                 input_pairs.append((query, doc_text)) # Tuple (query, passage)

            doc_map[idx] = doc # Lưu lại doc gốc theo index

        all_scores = []
        # Xử lý theo batch
        for i in range(0, len(input_pairs), self.batch_size):
            batch_input = input_pairs[i : i + self.batch_size]

            if self.model_type == 'qwen-reranker':
                # --- Qwen Reranker Logic ---
                inputs = self.tokenizer(
                    batch_input, padding=False, truncation='longest_first',
                    return_attention_mask=False,
                    max_length=self.max_length - len(self.prefix_tokens) - len(self.suffix_tokens)
                )
                # Add prefix/suffix
                for j in range(len(inputs['input_ids'])):
                    inputs['input_ids'][j] = self.prefix_tokens + inputs['input_ids'][j] + self.suffix_tokens

                # Pad batch
                inputs = self.tokenizer.pad(inputs, padding=True, return_tensors="pt", max_length=self.max_length)
                inputs = {key: val.to(self.device) for key, val in inputs.items()}

                logits = self.model(**inputs).logits[:, -1, :]
                true_logits = logits[:, self.token_true_id]
                false_logits = logits[:, self.token_false_id]
                batch_scores_tensor = torch.stack([false_logits, true_logits], dim=1)
                batch_scores_tensor = F.log_softmax(batch_scores_tensor, dim=1)
                scores = batch_scores_tensor[:, 1].exp().cpu().tolist() # Lấy xác suất của 'yes'

            elif self.model_type == 'cross-encoder':
                # --- Standard Cross-Encoder Logic ---
                # Check model config for max sequence length, default to 512 if not found
                ce_max_length = getattr(self.tokenizer, 'model_max_length', 512)
                inputs = self.tokenizer(
                    batch_input, padding=True, truncation=True,
                    return_tensors="pt", max_length=ce_max_length
                ).to(self.device)

                outputs = self.model(**inputs)
                logits = outputs.logits

                # Xử lý output logits:
                if self.num_labels == 1: # Thường là score trực tiếp
                    scores = logits.squeeze(-1).cpu().tolist()
                elif self.num_labels >= 2: # Lấy logit/probability của lớp 'relevant' (thường là index 1)
                     # Apply sigmoid for binary classification or softmax for multi-class interpretation if needed
                     # For simplicity, let's assume higher logit at index 1 means more relevant
                     # Hoặc có thể dùng softmax để có xác suất
                     probabilities = torch.softmax(logits, dim=-1)
                     scores = probabilities[:, 1].cpu().tolist() # Giả sử index 1 là relevant
                else: # Trường hợp lạ
                    scores = [0.0] * len(batch_input) # Default score nếu không xác định được

                # Áp dụng sigmoid nếu output là logit đơn (để đưa về khoảng 0-1) - tùy chọn
                # if self.num_labels == 1:
                #    scores = [1 / (1 + math.exp(-score)) for score in scores]

            else:
                scores = [0.0] * len(batch_input) # Fallback

            all_scores.extend(scores)

            # Giải phóng bộ nhớ GPU sau mỗi batch
            del inputs
            if 'outputs' in locals(): del outputs
            if 'logits' in locals(): del logits
            if self.device == 'cuda':
                torch.cuda.empty_cache()

        # Gán điểm số vào doc_map và sắp xếp
        scored_docs = []
        for idx, score in enumerate(all_scores):
            if idx in doc_map: # Kiểm tra xem index có hợp lệ không
                doc = doc_map[idx]
                doc['rerank_score'] = float(score) # Chuyển sang float chuẩn
                scored_docs.append(doc)

        # Sắp xếp theo rerank_score giảm dần
        reranked_docs = sorted(scored_docs, key=lambda x: x.get('rerank_score', 0.0), reverse=True)

        return reranked_docs[:k]

# Implementing retrievers


## BM25 retriever


In [4]:
from rank_bm25 import BM25Okapi

class BM25Retriever:
    def __init__(self, corpus):
        self.doc_ids = [doc["id"] for doc in corpus]
        tokenized_corpus = [doc["text"].split() for doc in tqdm(corpus, desc="Tokenizing corpus for BM25")]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def search(self, query, k=10):
        tokenized_query = query.split()
        scores = self.bm25.get_scores(tokenized_query)
        top_k_indices = np.argpartition(scores, -k)[-k:]
        top_k_sorted_indices = top_k_indices[np.argsort(scores[top_k_indices])][::-1]
        return [(self.doc_ids[i], float(scores[i])) for i in top_k_sorted_indices]

## Dense retriever


In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import os
import torch

class DenseRetriever:
    def __init__(self, corpus, model_name="sentence-transformers/all-MiniLM-L6-v2", device="cuda", embeddings_dir=None):
        """
        Khởi tạo DenseRetriever.
        Sẽ ưu tiên tải embeddings từ file .npy tương ứng với model_name.
        Nếu không, nó sẽ encode corpus và lưu kết quả.
        """
        if not torch.cuda.is_available() and device == "cuda":
            print("CUDA not available, switching to CPU. This will be slow.")
            device = "cpu"
        self.device = device
        self.model_name = model_name
        print(f"Initializing DenseRetriever with model: {self.model_name} on device: {self.device}")
        self.model = SentenceTransformer(model_name, device=self.device)
        self.doc_ids = [doc["id"] for doc in corpus]

        # Tạo đường dẫn file embeddings dựa trên tên model
        safe_model_name = model_name.replace('/', '_') # Thay thế '/' để tạo tên file hợp lệ
        self.embeddings_path = os.path.join(embeddings_dir, f"corpus_embeddings_{safe_model_name}_small.npy")
        os.makedirs(embeddings_dir, exist_ok=True) # Tạo thư mục nếu chưa có

        if os.path.exists(self.embeddings_path):
            print(f"Loading pre-computed embeddings from {self.embeddings_path}...")
            self.doc_vecs = np.load(self.embeddings_path)
            print("Embeddings loaded successfully.")
        else:
            print(f"No pre-computed embeddings found at {self.embeddings_path}. Encoding corpus...")
            texts = [doc["text"] for doc in corpus]
            # Sử dụng batch_size phù hợp với GPU memory
            encode_batch_size = 128 if 'cuda' in self.device else 32
            self.doc_vecs = self.model.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=True,
                convert_to_numpy=True,
                batch_size=encode_batch_size
            )
            print(f"Saving embeddings to {self.embeddings_path} for future use...")
            np.save(self.embeddings_path, self.doc_vecs)
            print(f"New embeddings saved.")

        # Chuyển doc_vecs sang GPU để tính toán nhanh hơn nếu dùng CUDA
        self.doc_vecs_gpu = None
        if self.device == "cuda":
            try:
                self.doc_vecs_gpu = torch.from_numpy(self.doc_vecs).to(self.device)
                print("Document embeddings moved to GPU.")
            except Exception as e:
                print(f"Failed to move embeddings to GPU: {e}. Calculations will use CPU numpy.")
                self.doc_vecs_gpu = None # Fallback to CPU if GPU transfer fails

    # (Hàm search giữ nguyên)
    def search(self, query, k=10):
        q_vec = self.model.encode([query], normalize_embeddings=True, convert_to_tensor=True, device=self.device)
        if self.doc_vecs_gpu is not None:
            sims = torch.matmul(q_vec, self.doc_vecs_gpu.T).cpu().numpy()[0]
        else:
            q_vec_np = q_vec.cpu().numpy()[0]
            sims = np.dot(self.doc_vecs, q_vec_np)

        # Lấy top k indices hiệu quả hơn
        # top_idx = np.argsort(sims)[-k:][::-1] # Cách cũ, chậm hơn cho K nhỏ
        if k < len(sims):
            top_idx = np.argpartition(sims, -k)[-k:] # Nhanh hơn cho K nhỏ
            top_k_sorted_indices = top_idx[np.argsort(sims[top_idx])][::-1]
        else: # Nếu k >= số lượng documents
             top_k_sorted_indices = np.argsort(sims)[::-1]

        return [(self.doc_ids[i], float(sims[i])) for i in top_k_sorted_indices]


    # (Hàm search_batch giữ nguyên, nhưng đảm bảo dùng self.device)
    def search_batch(self, queries, k=10, batch_size=64):
        results = []
        for i in tqdm(range(0, len(queries), batch_size), desc=f"Dense Batch Search ({self.model_name})"):
            queries_chunk = queries[i:i+batch_size]
            q_vecs_chunk = self.model.encode(
                queries_chunk,
                normalize_embeddings=True,
                show_progress_bar=False,
                convert_to_tensor=True,
                device=self.device
            )

            if self.doc_vecs_gpu is not None:
                sims_chunk = torch.matmul(q_vecs_chunk, self.doc_vecs_gpu.T)
                sims_chunk_cpu = sims_chunk.cpu().numpy()
            else:
                q_vecs_chunk_np = q_vecs_chunk.cpu().numpy()
                sims_chunk_cpu = np.dot(q_vecs_chunk_np, self.doc_vecs.T)

            for row in sims_chunk_cpu:
                if k < len(row):
                    top_idx = np.argpartition(row, -k)[-k:]
                    top_k_sorted_indices = top_idx[np.argsort(row[top_idx])][::-1]
                else:
                    top_k_sorted_indices = np.argsort(row)[::-1]
                results.append([(self.doc_ids[j], float(row[j])) for j in top_k_sorted_indices])

            # Giải phóng bộ nhớ GPU của chunk query hiện tại
            del q_vecs_chunk
            if self.device == 'cuda':
                torch.cuda.empty_cache()

        return results

# Evaluation


In [6]:
import math
import json
import numpy as np
from tqdm import tqdm

def recall_at_k(run, qrels, k):
    # run: list of (docid, score)
    relevant = {d for d, rel in qrels.items() if rel > 0}
    retrieved = {d for d, _ in run[:k]}
    if not relevant:
        return 0.0
    return len(relevant & retrieved) / len(relevant)

def dcg_at_k(run, qrels, k):
    import math
    return sum(
        (2**qrels.get(doc, 0) - 1) / math.log2(idx + 2)
        for idx, (doc, _) in enumerate(run[:k])
    )

def ndcg_at_k(run, qrels, k):
    ideal = sorted(qrels.values(), reverse=True)[:k]
    idcg = sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(ideal))
    if idcg == 0:
        return 0.0
    return dcg_at_k(run, qrels, k) / idcg

def normalize_scores(scores_dict):
    if not scores_dict: return {}
    vals = list(scores_dict.values())
    min_v, max_v = min(vals), max(vals)
    range_v = max_v - min_v
    if range_v == 0: # Tránh chia cho 0 nếu tất cả điểm số bằng nhau
        return {k: 0.5 for k in scores_dict} # Hoặc trả về 0 hoặc 1 tùy logic
    return {k: (v - min_v) / range_v for k, v in scores_dict.items()}

In [7]:
# (Giữ nguyên các hàm metric: recall_at_k, ndcg_at_k)
# (Giữ nguyên class Reranker, BM25Retriever, DenseRetriever)

def evaluate(retriever, queries, qrels, corpus_map,
             reranker=None, k=10, retrieval_k=50, batch_size=64, save_path=None,
             initial_runs_data=None): # Thêm tham số initial_runs_data
    """
    Đánh giá retriever hoặc reranker trên dữ liệu có sẵn.
    Nếu initial_runs_data được cung cấp, sẽ bỏ qua bước retrieval.
    """
    ndcgs, recalls = [], []
    all_final_outputs = []

    q_texts = [q["text"] for q in queries]
    q_ids = [q["id"] for q in queries]

    # Xác định tên cho log
    eval_name = ""
    if initial_runs_data is None: # Chạy retrieval
        eval_name = f"{retriever.__class__.__name__}"
    else: # Chạy reranking dựa trên data có sẵn
         # Cố gắng lấy tên retriever gốc từ data (nếu có) hoặc dùng tên chung
         base_retriever_name = "Precomputed"
         if initial_runs_data and isinstance(initial_runs_data, list) and initial_runs_data[0].get('retriever_info'):
              base_retriever_name = initial_runs_data[0]['retriever_info']
         eval_name = f"{base_retriever_name}"

    if reranker:
        eval_name += f" + {reranker.model_name.split('/')[-1]}" # Thêm tên reranker nếu có

    print(f"--- Starting Evaluation for: {eval_name} ---")

    all_initial_runs_tuples = [] # List[List[tuple(doc_id, score)]]

    # === Bước 1: Initial Retrieval (HOẶC Load dữ liệu có sẵn) ===
    start_retrieval_time = time.time()
    if initial_runs_data is None:
        # --- Chạy retrieval ---
        current_retrieval_k = retrieval_k if reranker else k
        print(f"Step 1: Retrieving top {current_retrieval_k} candidates for {len(queries)} queries using {retriever.__class__.__name__}...")

        if isinstance(retriever, BM25Retriever):
            for qtext in tqdm(q_texts, desc=f"BM25 Search (k={current_retrieval_k})"):
                all_initial_runs_tuples.append(retriever.search(qtext, k=current_retrieval_k))
        elif isinstance(retriever, DenseRetriever):
             all_initial_runs_tuples = retriever.search_batch(q_texts, k=current_retrieval_k, batch_size=batch_size)
        else:
            raise TypeError("Unsupported retriever type")

        retrieval_time = time.time() - start_retrieval_time
        print(f"Initial retrieval completed in {retrieval_time:.2f} seconds.")

        # Lưu thông tin retriever vào output để dùng sau (tùy chọn)
        # Sẽ hữu ích nếu hàm evaluate trả về cả raw_outputs
        raw_outputs_to_save = []
        for i in range(len(queries)):
             raw_outputs_to_save.append({
                 "query_id": q_ids[i],
                 "query_text": q_texts[i],
                 "retriever_info": retriever.__class__.__name__, # Lưu tên retriever
                 "results": [{"doc_id": docid, "score": score} for docid, score in all_initial_runs_tuples[i]]
             })

    else:
        # --- Load dữ liệu có sẵn ---
        print(f"Step 1: Loading pre-computed initial runs for {len(queries)} queries...")
        # initial_runs_data phải là list of dict giống format all_outputs
        all_initial_runs_tuples = []
        for run_data in initial_runs_data:
             # Lấy tối đa retrieval_k kết quả từ data có sẵn
             results_list = run_data.get("results", [])[:retrieval_k]
             all_initial_runs_tuples.append([(res["doc_id"], res.get("score", 0.0)) for res in results_list])
        retrieval_time = 0 # Không tốn thời gian retrieval
        print("Pre-computed runs loaded.")
        raw_outputs_to_save = initial_runs_data # Giữ nguyên dữ liệu gốc nếu dùng cho rerank

    # === Bước 2: Reranking (Nếu có) ===
    start_rerank_time = time.time()
    if reranker:
        print(f"Step 2: Reranking top {retrieval_k} candidates for each query using {reranker.model_name}...")
        all_final_runs_tuples = [] # List[List[tuple(doc_id, rerank_score)]]
        num_queries = len(queries)

        for i in tqdm(range(num_queries), desc=f"Reranking ({reranker.model_name.split('/')[-1]})"):
             qid = q_ids[i]
             qtext = q_texts[i]
             initial_run_tuples = all_initial_runs_tuples[i]
             qrel = qrels.get(qid, {})

             docs_to_rerank = [{"doc_id": docid, "text": corpus_map.get(docid, ""), "initial_score": score}
                               for docid, score in initial_run_tuples]

             # Nếu docs_to_rerank rỗng, bỏ qua
             if not docs_to_rerank:
                 all_final_runs_tuples.append([])
                 all_final_outputs.append({
                     "query_id": qid, "query_text": qtext, "results": []
                 })
                 continue

             reranked_run_full = reranker.rerank(qtext, docs_to_rerank, k=k)

             final_run_tuples = [(doc['doc_id'], doc['rerank_score']) for doc in reranked_run_full]
             all_final_runs_tuples.append(final_run_tuples)

             all_final_outputs.append({
                 "query_id": qid,
                 "query_text": qtext,
                 "retriever_info": raw_outputs_to_save[i].get('retriever_info', 'Precomputed'), # Giữ thông tin retriever gốc
                 "results": reranked_run_full
             })

        rerank_time = time.time() - start_rerank_time
        avg_latency_ms = (rerank_time / num_queries) * 1000 if num_queries > 0 else 0
        print(f"--- Reranking Performance ({reranker.model_name}) ---")
        print(f"Total time for reranking {num_queries} queries: {rerank_time:.2f} seconds")
        print(f"Average reranking latency: {avg_latency_ms:.2f} ms/query")

    else:
        # Không rerank, kết quả cuối cùng là initial run (đã cắt k)
        print("Step 2: No reranker specified. Using initial retrieval results.")
        all_final_runs_tuples = [run[:k] for run in all_initial_runs_tuples] # Cắt top k
        rerank_time = 0
        all_final_outputs = raw_outputs_to_save # Giữ nguyên raw outputs

    # === Bước 3: Tính toán Metrics ===
    print("Step 3: Calculating metrics...")
    for i in range(len(queries)):
        qid = q_ids[i]
        final_run_tuples = all_final_runs_tuples[i]
        qrel = qrels.get(qid, {})
        ndcgs.append(ndcg_at_k(final_run_tuples, qrel, k))
        recalls.append(recall_at_k(final_run_tuples, qrel, k))

    # === Bước 4: Lưu kết quả ===
    if save_path:
        # Tạo thư mục nếu chưa tồn tại
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        print(f"Saving {len(all_final_outputs)} final results to {save_path}...")
        try:
            with open(save_path, "w", encoding="utf-8") as f:
                for record in all_final_outputs:
                    # Đảm bảo score là float chuẩn
                    if 'results' in record:
                        for res in record['results']:
                            if 'score' in res: res['score'] = float(res['score'])
                            if 'rerank_score' in res: res['rerank_score'] = float(res['rerank_score'])
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
            print("Saved successfully.")
        except Exception as e:
            print(f"Error saving results to {save_path}: {e}")

    # Trả về kết quả trung bình và dữ liệu output thô (quan trọng cho các bước sau)
    mean_ndcg = float(np.mean(ndcgs)) if ndcgs else 0.0
    mean_recall = float(np.mean(recalls)) if recalls else 0.0

    print(f"--- Final Metrics for {eval_name} (k={k}) ---")
    print(f"Average nDCG@{k}: {mean_ndcg:.4f}")
    print(f"Average Recall@{k}: {mean_recall:.4f}")
    print("-" * 30)

    # Nếu không rerank, trả về raw_outputs_to_save để dùng cho hybrid/rerank sau
    # Nếu có rerank, trả về all_final_outputs (đã rerank)
    return mean_ndcg, mean_recall, all_final_outputs

In [8]:
def combine_and_evaluate_hybrid(bm25_results_path, dense_results_path, qrels, corpus_map, k=10, save_path=None, method='rrf', alpha=0.5, rrf_k=60):
    """
    Load kết quả BM25 và Dense từ file, trộn chúng lại, tính metric, lưu kết quả hybrid-raw,
    và trả về dữ liệu hybrid-raw (list of dicts).
    """
    print(f"\n--- Combining BM25 ({bm25_results_path}) and Dense ({dense_results_path}) using '{method}' ---")

    # Load data
    try:
        bm25_outputs = load_jsonl(bm25_results_path)
        dense_outputs = load_jsonl(dense_results_path)
    except FileNotFoundError as e:
        print(f"Error loading results file: {e}. Cannot proceed with hybrid combination.")
        return None, None, [] # Trả về None nếu không load được file

    assert len(bm25_outputs) == len(dense_outputs), "Number of queries in BM25 and Dense outputs do not match!"

    ndcgs, recalls = [], []
    hybrid_outputs_raw = [] # Dữ liệu hybrid thô (sau khi trộn)

    # Lấy tên của dense model từ file path hoặc data (nếu có)
    dense_model_name_inferred = "UnknownDense"
    if dense_outputs and dense_outputs[0].get('retriever_info'):
        dense_model_name_inferred = dense_outputs[0]['retriever_info']
    elif "MiniLM" in dense_results_path:
        dense_model_name_inferred = "DenseRetriever_MiniLM"
    elif "bge" in dense_results_path:
         dense_model_name_inferred = "DenseRetriever_BGE"

    hybrid_retriever_name = f"Hybrid_{method}_{dense_model_name_inferred}"


    for i in tqdm(range(len(bm25_outputs)), desc=f"Combining Hybrid ({method})"):
        bm_res = bm25_outputs[i]
        dn_res = dense_outputs[i]

        qid = bm_res["query_id"]
        qtext = bm_res["query_text"]
        qrel = qrels.get(qid, {})

        # --- Logic kết hợp (Giữ nguyên) ---
        run_tuples = [] # List of (doc_id, combined_score)
        # Lấy kết quả từ dict (có thể chỉ có score, ko có text)
        bm_results_list = bm_res.get('results', [])
        dn_results_list = dn_res.get('results', [])

        if method == 'weighted_sum':
            bm_dict = {r['doc_id']: r['score'] for r in bm_results_list}
            dn_dict = {r['doc_id']: r['score'] for r in dn_results_list}
            bm_norm = normalize_scores(bm_dict)
            dn_norm = normalize_scores(dn_dict)
            all_ids = set(bm_norm.keys()) | set(dn_norm.keys())
            scores = {
                doc_id: alpha * dn_norm.get(doc_id, 0) + (1 - alpha) * bm_norm.get(doc_id, 0)
                for doc_id in all_ids
            }
            run_tuples = sorted(scores.items(), key=lambda item: item[1], reverse=True)

        elif method == 'rrf':
            bm_ranks = {r['doc_id']: i + 1 for i, r in enumerate(bm_results_list)}
            dn_ranks = {r['doc_id']: i + 1 for i, r in enumerate(dn_results_list)}
            all_ids = set(bm_ranks.keys()) | set(dn_ranks.keys())
            rrf_scores = {}
            for doc_id in all_ids:
                score = 0.0
                if doc_id in bm_ranks: score += 1.0 / (rrf_k + bm_ranks[doc_id])
                if doc_id in dn_ranks: score += 1.0 / (rrf_k + dn_ranks[doc_id])
                rrf_scores[doc_id] = score
            run_tuples = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
        else:
            raise ValueError("Method must be 'weighted_sum' or 'rrf'")

        # Cắt top K cho việc tính metric raw
        run_tuples_k = run_tuples[:k]

        ndcgs.append(ndcg_at_k(run_tuples_k, qrel, k))
        recalls.append(recall_at_k(run_tuples_k, qrel, k))

        # Lưu lại TOÀN BỘ kết quả đã trộn (không cắt k) để reranker có thể dùng
        hybrid_outputs_raw.append({
            "query_id": qid,
            "query_text": qtext,
            "retriever_info": hybrid_retriever_name, # Đặt tên cho retriever hybrid
            # Lưu ý: Score ở đây là score đã được trộn (combined score)
            "results": [{"doc_id": docid, "score": score} for docid, score in run_tuples]
        })

    mean_ndcg = float(np.mean(ndcgs)) if ndcgs else 0.0
    mean_recall = float(np.mean(recalls)) if recalls else 0.0

    print(f"--- Hybrid Raw Metrics ({method}, k={k}) ---")
    print(f"Average nDCG@{k}: {mean_ndcg:.4f}")
    print(f"Average Recall@{k}: {mean_recall:.4f}")

    # Lưu kết quả hybrid-raw (chứa toàn bộ danh sách đã trộn)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        print(f"Saving {len(hybrid_outputs_raw)} raw hybrid results to {save_path}...")
        try:
             save_jsonl(hybrid_outputs_raw, save_path) # Dùng hàm save_jsonl đã có
             print(f"Saved raw hybrid results successfully.")
        except Exception as e:
             print(f"Error saving raw hybrid results to {save_path}: {e}")

    return mean_ndcg, mean_recall, hybrid_outputs_raw # Trả về dữ liệu để dùng cho rerank

## PHASE 0: PREPARATION


In [11]:
# corpus = load_jsonl(os.path.join(base_path, "corpus.jsonl"))
# queries = load_jsonl(os.path.join(base_path, "queries.jsonl"))
# qrels = load_qrels(os.path.join(base_path, "qrels.tsv"))

# corpus_map = {doc['id']: doc['text'] for doc in corpus}

# NUM_QUERY=1000

# # sử dụng seed 42 chọn ra 200 query
# import random
# random.seed(42)
# queries = random.sample(queries, NUM_QUERY)

# results_dir = os.path.join(base_path, "results_optimized") # Thư mục lưu kết quả mới
# os.makedirs(results_dir, exist_ok=True)
import random

NUM_CORPUS_TARGET = 50000 # Kích thước Corpus mục tiêu
NUM_QUERY = 1000          # Số lượng query cần đánh giá

# --- 1. Tải Dữ liệu Gốc ---
corpus = load_jsonl(os.path.join(base_path, "corpus.jsonl"))
queries_full = load_jsonl(os.path.join(base_path, "queries.jsonl"))
qrels_full = load_qrels(os.path.join(base_path, "qrels.tsv"))

print(f"Original Corpus size: {len(corpus):,}")
print(f"Original Queries size: {len(queries_full):,}")

# --- 2. Chọn 1000 Queries và Qrels tương ứng ---
random.seed(42)
if len(queries_full) > NUM_QUERY:
    queries_eval = random.sample(queries_full, NUM_QUERY)
else:
    queries_eval = queries_full # Nếu ít hơn 1000 thì dùng hết

qids_eval = {q['id'] for q in queries_eval}

# Lọc Qrels để chỉ chứa các entries cho 1000 queries đã chọn
qrels_eval = {qid: docs for qid, docs in qrels_full.items() if qid in qids_eval}

# Lấy tất cả Document IDs relevant (ground truth)
relevant_doc_ids = set()
for docs in qrels_eval.values():
    relevant_doc_ids.update({docid for docid, rel in docs.items() if rel > 0})

# Loại bỏ các query không còn relevant doc nào sau khi lọc (thực tế ít xảy ra)
qids_with_relevant_doc = {qid for qid, docs in qrels_eval.items() if docs}
queries_eval = [q for q in queries_eval if q['id'] in qids_with_relevant_doc]

# --- 3. Chọn Noise Documents để đạt 50,000 Corpus ---
corpus_map_full = {doc['id']: doc for doc in corpus}
all_doc_ids = set(corpus_map_full.keys())
noise_candidate_ids = all_doc_ids - relevant_doc_ids

num_relevant_docs = len(relevant_doc_ids)
required_noise = max(0, NUM_CORPUS_TARGET - num_relevant_docs)

random.seed(42) # Giữ seed để random noise
# Đảm bảo không lấy nhiều hơn số lượng có sẵn
num_noise_to_take = min(required_noise, len(noise_candidate_ids))

if num_noise_to_take > 0:
    noise_doc_ids = random.sample(list(noise_candidate_ids), num_noise_to_take)
else:
    noise_doc_ids = []

# --- 4. Xây dựng Corpus và Qrels Cuối cùng ---
final_corpus_ids = relevant_doc_ids.union(set(noise_doc_ids))

# Lọc corpus cuối cùng
corpus_subset = [corpus_map_full[doc_id] for doc_id in final_corpus_ids]

# Cập nhật các biến cho bước tiếp theo
queries = queries_eval
qrels = qrels_eval
corpus = corpus_subset

# Dữ liệu cần thiết cho việc index Milvus
corpus_texts = [doc['text'] for doc in corpus]
corpus_ids = [doc['id'] for doc in corpus]
corpus_map = {doc['id']: doc['text'] for doc in corpus} # Cần cập nhật corpus_map

print("-" * 50)
print(f"Final Corpus size: {len(corpus):,} documents (Relevant: {num_relevant_docs:,}, Noise: {len(noise_doc_ids):,})")
print(f"Final Queries size: {len(queries):,} queries (Tất cả đều có relevant doc trong corpus)")
print(f"Final Qrels size: {sum(len(v) for v in qrels.values()):,} relevant entries")
print("-" * 50)

results_dir = os.path.join(base_path, "results_optimized_2")
os.makedirs(results_dir, exist_ok=True)

# NUM_QUERY=1000

# # sử dụng seed 42 chọn ra 200 query
# import random
# random.seed(42)
# queries = random.sample(queries, NUM_QUERY)

# results_dir = os.path.join(base_path, "results_optimized")
# os.makedirs(results_dir, exist_ok=True)

corpus_texts = [doc['text'] for doc in corpus]
corpus_ids = [doc['id'] for doc in corpus]

results = {} # Dict lưu trữ metrics cuối cùng

Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/corpus.jsonl
Successfully loaded 1008985 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/corpus.jsonl
Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/queries.jsonl
Successfully loaded 101093 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/queries.jsonl
Original Corpus size: 1,008,985
Original Queries size: 101,093
--------------------------------------------------
Final Corpus size: 50,000 documents (Relevant: 595, Noise: 49,405)
Final Queries size: 556 queries (Tất cả đều có relevant doc trong corpus)
Final Qrels size: 595 relevant entries
--------------------------------------------------


## PHASE 1: INDIVIDUAL MODELS


### Sparse: BM25


In [45]:
SAVE_PATH = os.path.join(results_dir, f"bm25_raw_k100_q{NUM_QUERY}.jsonl")
KEY = "BM25 Raw (k=10)"

bm25_raw_outputs = None # Biến lưu kết quả thô cuối cùng

# --- Logic Kiểm tra File và Load ---
if os.path.exists(SAVE_PATH):
    print(f"\n--- Loading EXISTING BM25 Raw Results from {SAVE_PATH} ---")

    bm25_raw_outputs = load_jsonl(SAVE_PATH)

    # 3. Tính lại metrics trung bình từ dữ liệu đã load (sử dụng k=10)
    ndcgs, recalls = [], []
    k_final = 3

    for record in bm25_raw_outputs:
        q_id = record["query_id"]
        q_rel = qrels.get(q_id, {})
        # Lấy list (doc_id, score) từ output
        final_run = [(d['doc_id'], d.get('score', 0.0)) for d in record['results']]

        # Tính metrics
        ndcgs.append(ndcg_at_k(final_run, q_rel, k_final))
        recalls.append(recall_at_k(final_run, q_rel, k_final))

    mean_ndcg = float(np.mean(ndcgs)) if ndcgs else 0.0
    mean_recall = float(np.mean(recalls)) if recalls else 0.0

    print(f"--- Loaded Metrics (k=10) ---")
    print(f"Average nDCG@{k_final}: {mean_ndcg:.4f}")
    print(f"Average Recall@{k_final}: {mean_recall:.4f}")
    print("-" * 30)

    results[KEY] = {"nDCG@10": mean_ndcg, "Recall@10": mean_recall}
else:
    bm25_retriever = BM25Retriever(corpus)
    print("\n--- Evaluating BM25 Raw ---")
    ndcg_100, recall_100, bm25_raw_outputs = evaluate(
        bm25_retriever, queries, qrels, corpus_map, k=30, # k=100 (lưu data và tính metrics @100)
        save_path=os.path.join(results_dir, f"bm25_raw_k100_q{NUM_QUERY}.jsonl")
    )

    # 2. Tính lại metrics trung bình từ dữ liệu đã load/tạo ra (sử dụng k=10)
    ndcgs, recalls = [], []
    k_final = 3 # <-- Đặt k cho metrics là 10

    for record in bm25_raw_outputs:
        q_id = record["query_id"]
        q_rel = qrels.get(q_id, {})

        # Lấy list (doc_id, score) từ output
        # Lưu ý: 'results' chứa 100 kết quả
        final_run = [(d['doc_id'], d.get('score', 0.0)) for d in record['results']]

        # Tính metrics chỉ trên Top 10 của danh sách 100 kết quả
        ndcgs.append(ndcg_at_k(final_run, q_rel, k_final))
        recalls.append(recall_at_k(final_run, q_rel, k_final))

    mean_ndcg = float(np.mean(ndcgs)) if ndcgs else 0.0
    mean_recall = float(np.mean(recalls)) if recalls else 0.0

    print(f"--- Calculated Metrics (k=3) ---")
    print(f"Average nDCG@{k_final}: {mean_ndcg:.4f}")
    print(f"Average Recall@{k_final}: {mean_recall:.4f}")
    print("-" * 30)

    # 3. Lưu kết quả trung bình @10
    results["BM25 Raw (k=3)"] = {f"nDCG@{k_final}": mean_ndcg, f"Recall@{k_final}": mean_recall}

    del bm25_retriever
    gc.collect()


--- Loading EXISTING BM25 Raw Results from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl ---
Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl
Successfully loaded 556 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl
--- Loaded Metrics (k=10) ---
Average nDCG@3: 0.2821
Average Recall@3: 0.3085
------------------------------


### Dense: MiniLM-L6


In [46]:
dense_minilm = None
minilm_raw_outputs = None
try:
    print("\n--- Evaluating Dense MiniLM Raw ---")
    dense_minilm = DenseRetriever(corpus, model_name="sentence-transformers/all-MiniLM-L6-v2", device="cuda", embeddings_dir=base_path)

    # BƯỚC 1: Chạy evaluate (Lấy Top 100, metrics trả về là @100, nhưng ta sẽ bỏ qua)
    ndcg_100, recall_100, minilm_raw_outputs = evaluate(
        dense_minilm, queries, qrels, corpus_map, k=30,
        batch_size=128,
        save_path=os.path.join(results_dir, f"dense_minilm_raw_k100_q{NUM_QUERY}.jsonl")
    )

    # BƯỚC 2: Tính lại metrics trung bình nDCG@10 và Recall@10 trên toàn bộ outputs
    ndcgs, recalls = [], []
    k_final = 3

    for record in minilm_raw_outputs: # Lặp qua TẤT CẢ truy vấn
        q_id = record["query_id"]
        q_rel = qrels.get(q_id, {})
        final_run = [(d['doc_id'], d.get('score', 0.0)) for d in record['results']]

        ndcgs.append(ndcg_at_k(final_run, q_rel, k_final))
        recalls.append(recall_at_k(final_run, q_rel, k_final))

    mean_ndcg_3 = float(np.mean(ndcgs)) if ndcgs else 0.0
    mean_recall_3 = float(np.mean(recalls)) if recalls else 0.0

    print(f"--- MiniLM Calculated Metrics (k=3) ---")
    print(f"Average nDCG@{k_final}: {mean_ndcg_3:.4f}")
    print(f"Average Recall@{k_final}: {mean_recall_3:.4f}")

    results["Dense MiniLM Raw (k=3)"] = {f"nDCG@{k_final}": mean_ndcg_3, f"Recall@{k_final}": mean_recall_3}

except Exception as e:
    print(f"FAILED to run Dense MiniLM Raw: {e}")

del dense_minilm
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


--- Evaluating Dense MiniLM Raw ---
Initializing DenseRetriever with model: sentence-transformers/all-MiniLM-L6-v2 on device: cuda
Loading pre-computed embeddings from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/corpus_embeddings_sentence-transformers_all-MiniLM-L6-v2_small.npy...
Embeddings loaded successfully.
Document embeddings moved to GPU.
--- Starting Evaluation for: DenseRetriever ---
Step 1: Retrieving top 30 candidates for 556 queries using DenseRetriever...


Dense Batch Search (sentence-transformers/all-MiniLM-L6-v2): 100%|██████████| 5/5 [00:00<00:00, 11.87it/s]


Initial retrieval completed in 0.42 seconds.
Step 2: No reranker specified. Using initial retrieval results.
Step 3: Calculating metrics...
Saving 556 final results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/dense_minilm_raw_k100_q1000.jsonl...
Saved successfully.
--- Final Metrics for DenseRetriever (k=30) ---
Average nDCG@30: 0.8920
Average Recall@30: 0.9874
------------------------------
--- MiniLM Calculated Metrics (k=3) ---
Average nDCG@3: 0.8658
Average Recall@3: 0.9125


### Dense: BGE-small-en


In [47]:
dense_bge_small = None
bge_small_raw_outputs = None

try:
    print("\n--- Evaluating Dense BGE-Small Raw ---")
    dense_minilm = DenseRetriever(corpus, model_name="BAAI/bge-small-en-v1.5", device="cuda", embeddings_dir=base_path)

    # BƯỚC 1: Chạy evaluate (Lấy Top 100, metrics trả về là @100, nhưng ta sẽ bỏ qua)
    ndcg_100, recall_100, bge_small_raw_outputs = evaluate(
        dense_minilm, queries, qrels, corpus_map, k=30,
        batch_size=128,
        save_path=os.path.join(results_dir, f"dense_bge_small_raw_k100_q{NUM_QUERY}.jsonl")
    )

    # BƯỚC 2: Tính lại metrics trung bình nDCG@10 và Recall@10 trên toàn bộ outputs
    ndcgs, recalls = [], []
    k_final = 3

    for record in bge_small_raw_outputs: # Lặp qua TẤT CẢ truy vấn
        q_id = record["query_id"]
        q_rel = qrels.get(q_id, {})
        final_run = [(d['doc_id'], d.get('score', 0.0)) for d in record['results']]

        ndcgs.append(ndcg_at_k(final_run, q_rel, k_final))
        recalls.append(recall_at_k(final_run, q_rel, k_final))

    mean_ndcg_3 = float(np.mean(ndcgs)) if ndcgs else 0.0
    mean_recall_3 = float(np.mean(recalls)) if recalls else 0.0

    print(f"--- BGE-Small Calculated Metrics (k=10) ---")
    print(f"Average nDCG@{k_final}: {mean_ndcg_3:.4f}")
    print(f"Average Recall@{k_final}: {mean_recall_3:.4f}")

    results["Dense BGE-Small Raw (k=3)"] = {f"nDCG@{k_final}": mean_ndcg_3, f"Recall@{k_final}": mean_recall_3}

except Exception as e:
    print(f"FAILED to run Dense BGE-Small Raw: {e}")

del dense_bge_small
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Cleanup for Dense BGE-Small complete.")


--- Evaluating Dense BGE-Small Raw ---
Initializing DenseRetriever with model: BAAI/bge-small-en-v1.5 on device: cuda
Loading pre-computed embeddings from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/corpus_embeddings_BAAI_bge-small-en-v1.5_small.npy...
Embeddings loaded successfully.
Document embeddings moved to GPU.
--- Starting Evaluation for: DenseRetriever ---
Step 1: Retrieving top 30 candidates for 556 queries using DenseRetriever...


Dense Batch Search (BAAI/bge-small-en-v1.5): 100%|██████████| 5/5 [00:00<00:00, 12.52it/s]


Initial retrieval completed in 0.40 seconds.
Step 2: No reranker specified. Using initial retrieval results.
Step 3: Calculating metrics...
Saving 556 final results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/dense_bge_small_raw_k100_q1000.jsonl...
Saved successfully.
--- Final Metrics for DenseRetriever (k=30) ---
Average nDCG@30: 0.9083
Average Recall@30: 0.9937
------------------------------
--- BGE-Small Calculated Metrics (k=10) ---
Average nDCG@3: 0.8855
Average Recall@3: 0.9257
Cleanup for Dense BGE-Small complete.


In [48]:
# --- Kiểm tra xem các file kết quả thô có tồn tại không ---
bm25_raw_path = os.path.join(results_dir, f"bm25_raw_k100_q{NUM_QUERY}.jsonl")
minilm_raw_path = os.path.join(results_dir, f"dense_minilm_raw_k100_q{NUM_QUERY}.jsonl")
bge_small_raw_path = os.path.join(results_dir, f"dense_bge_small_raw_k100_q{NUM_QUERY}.jsonl")

if not os.path.exists(bm25_raw_path): print(f"WARNING: BM25 raw results file not found: {bm25_raw_path}")
if not os.path.exists(minilm_raw_path): print(f"WARNING: MiniLM raw results file not found: {minilm_raw_path}")
if not os.path.exists(bge_small_raw_path): print(f"WARNING: BGE-Small raw results file not found: {bge_small_raw_path}")

## PHASE 2: HYBRID RESULTS (MODEL-FREE)


In [49]:
hybrid_minilm_rrf_outputs = []
hybrid_bge_small_rrf_outputs = []

# --- 4. Hybrid BM25 + MiniLM (RRF) Raw ---
if os.path.exists(bm25_raw_path) and os.path.exists(minilm_raw_path):
    hybrid_minilm_path = os.path.join(results_dir, f"hybrid_minilm_rrf_raw_k100_q{NUM_QUERY}.jsonl")
    ndcg, recall, hybrid_minilm_rrf_outputs = combine_and_evaluate_hybrid(
        bm25_raw_path, minilm_raw_path, qrels, corpus_map, k=3, # Tính metric k=10 cho raw hybrid
        save_path=hybrid_minilm_path, # Lưu kết quả hybrid thô (k=100+)
        method='rrf', rrf_k=60
    )
    if ndcg is not None:
         results["Hybrid MiniLM RRF Raw (k=10)"] = {"nDCG@10": ndcg, "Recall@10": recall}
    else:
         hybrid_minilm_rrf_outputs = [] # Đảm bảo là list rỗng nếu lỗi
else:
    print("Skipping Hybrid MiniLM RRF Raw due to missing input files.")

# --- 5. Hybrid BM25 + BGE-Small (RRF) Raw ---
if os.path.exists(bm25_raw_path) and os.path.exists(bge_small_raw_path):
    hybrid_bge_path = os.path.join(results_dir, f"hybrid_bge_small_rrf_raw_k100_q{NUM_QUERY}.jsonl")
    ndcg, recall, hybrid_bge_small_rrf_outputs = combine_and_evaluate_hybrid(
        bm25_raw_path, bge_small_raw_path, qrels, corpus_map, k=3, # Tính metric k=10 cho raw hybrid
        save_path=hybrid_bge_path, # Lưu kết quả hybrid thô (k=100+)
        method='rrf', rrf_k=60
    )
    if ndcg is not None:
        results["Hybrid BGE-Small RRF Raw (k=10)"] = {"nDCG@10": ndcg, "Recall@10": recall}
    else:
        hybrid_bge_small_rrf_outputs = [] # Đảm bảo là list rỗng nếu lỗi
else:
    print("Skipping Hybrid BGE-Small RRF Raw due to missing input files.")

print("\n--- Retrieval and Fusion Phase Complete. VRAM should be clear. ---")


--- Combining BM25 (/content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl) and Dense (/content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/dense_minilm_raw_k100_q1000.jsonl) using 'rrf' ---
Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl
Successfully loaded 556 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl
Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/dense_minilm_raw_k100_q1000.jsonl
Successfully loaded 556 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/dense_minilm_raw_k100_q1000.jsonl


Combining Hybrid (rrf): 100%|██████████| 556/556 [00:00<00:00, 8378.48it/s]

--- Hybrid Raw Metrics (rrf, k=3) ---
Average nDCG@3: 0.5786
Average Recall@3: 0.6885
Saving 556 raw hybrid results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/hybrid_minilm_rrf_raw_k100_q1000.jsonl...


Saved raw hybrid results successfully.

--- Combining BM25 (/content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl) and Dense (/content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/dense_bge_small_raw_k100_q1000.jsonl) using 'rrf' ---
Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl
Successfully loaded 556 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/bm25_raw_k100_q1000.jsonl
Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/dense_bge_small_raw_k100_q1000.jsonl
Successfully loaded 556 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/dense_bge_small_raw_k100_q1000.jsonl


Combining Hybrid (rrf): 100%|██████████| 556/556 [00:00<00:00, 8714.82it/s]

--- Hybrid Raw Metrics (rrf, k=3) ---
Average nDCG@3: 0.5830
Average Recall@3: 0.6787
Saving 556 raw hybrid results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/hybrid_bge_small_rrf_raw_k100_q1000.jsonl...
Saved raw hybrid results successfully.

--- Retrieval and Fusion Phase Complete. VRAM should be clear. ---


## PHASE 3: SEQUENTIAL RERANKINGS


In [50]:
datasets_to_rerank = {
    # Key dùng để đặt tên file và lưu kết quả
    "BM25_Raw": bm25_raw_outputs,
    "Dense_MiniLM_Raw": minilm_raw_outputs,
    "Dense_BGE-Small_Raw": bge_small_raw_outputs,
    "Hybrid_MiniLM_RRF_Raw": hybrid_minilm_rrf_outputs,
    "Hybrid_BGE-Small_RRF_Raw": hybrid_bge_small_rrf_outputs,
}

RERANKER_CONFIGS = [
    # Qwen3-4B (~8GB+ VRAM)
    # {"name": "Qwen-4B", "model_name": "Qwen/Qwen3-Reranker-4B", "model_type": "qwen-reranker", "batch_size": 4},
    # BGE Reranker (~2.6GB+)
    {"name": "BGE-Large", "model_name": "BAAI/bge-reranker-large", "model_type": "cross-encoder", "batch_size": 64},
    # # # MiniLM-L12 Cross-Encoder (~0.5GB)
    {"name": "MiniLM-L12", "model_name": "cross-encoder/ms-marco-MiniLM-L-12-v2", "model_type": "cross-encoder", "batch_size": 64},
]

In [51]:
# Load 1 Reranker -> Chạy 5 Tests -> Unload
for reranker_cfg in RERANKER_CONFIGS:
    reranker_name = reranker_cfg['name']
    print(f"\n===== PROCESSING RERANKER: {reranker_name} =====")
    reranker_obj = None

    try:
        # 1. Load Reranker hiện tại (CHỈ LOAD 1 LẦN)
        print(f"Loading {reranker_name}...")
        reranker_obj = Reranker(
            model_name=reranker_cfg['model_name'],
            model_type=reranker_cfg['model_type'],
            batch_size=reranker_cfg.get('batch_size', 4),
            device="cuda"
        )

        # 2. Lặp qua từng bộ dữ liệu thô ĐỂ RERANK VỚI MODEL HIỆN TẠI
        for data_key_prefix, initial_data in datasets_to_rerank.items():

            # Kiểm tra xem dữ liệu thô có hợp lệ không
            if not initial_data or not isinstance(initial_data, list) or len(initial_data) == 0:
                print(f"Skipping reranking for {data_key_prefix} with {reranker_name} - Initial data is missing or empty.")
                continue # Bỏ qua bộ dữ liệu này nếu rỗng

            # Tạo key kết quả cuối cùng
            reranked_key = f"{data_key_prefix.replace('_Raw', '')} + {reranker_name}" # Ví dụ: "BM25 + Qwen-4B"
            print(f"\n--- Reranking: {reranked_key} ---")

            # Gọi evaluate ở chế độ rerank, truyền reranker_obj và initial_data
            # Hàm evaluate sẽ KHÔNG load lại model, chỉ sử dụng reranker_obj đã load
            ndcg, recall, _ = evaluate(
                retriever=None, # Không cần retriever ở đây
                queries=queries, # Sử dụng list queries đã load
                qrels=qrels,
                corpus_map=corpus_map,
                reranker=reranker_obj, # Truyền reranker đã load
                k=3,
                retrieval_k=30, # Reranker xử lý top 100 từ dữ liệu thô
                initial_runs_data=initial_data, # Dữ liệu thô làm đầu vào
                # Lưu file kết quả cho từng thử nghiệm rerank
                save_path=os.path.join(results_dir, f"{reranked_key.replace(' ', '_').replace('+', '_')}.jsonl")
            )
            # Lưu metric vào dict tổng
            results[reranked_key] = {"nDCG@10": ndcg, "Recall@10": recall}

    except Exception as e:
        # Ghi nhận lỗi nếu load hoặc chạy reranker thất bại
        print(f"!! FAILED during processing with Reranker {reranker_name}: {e}")
        # Ghi nhận lỗi cho tất cả các key liên quan đến reranker này (nếu cần)
        for data_key_prefix in datasets_to_rerank.keys():
             failed_key = f"{data_key_prefix.replace('_Raw', '')} + {reranker_name}"
             if failed_key not in results: # Chỉ ghi lỗi nếu chưa có kết quả
                 results[failed_key] = {"nDCG@10": "LOAD/RUN ERROR", "Recall@10": "LOAD/RUN ERROR"}

    finally:
        if reranker_obj and hasattr(reranker_obj, 'model'):
            print(f"\nUnloading {reranker_name}...")
            del reranker_obj
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"Cleanup for {reranker_name} complete.")
        print("="*50)


===== PROCESSING RERANKER: BGE-Large =====
Loading BGE-Large...
Loading tokenizer for BAAI/bge-reranker-large...
Loading model BAAI/bge-reranker-large (cross-encoder)...
Cross-encoder loaded with 1 output labels.
Reranker (BAAI/bge-reranker-large) initialized successfully on cuda.

--- Reranking: BM25 + BGE-Large ---
--- Starting Evaluation for: BM25Retriever + bge-reranker-large ---
Step 1: Loading pre-computed initial runs for 556 queries...
Pre-computed runs loaded.
Step 2: Reranking top 30 candidates for each query using BAAI/bge-reranker-large...


Reranking (bge-reranker-large): 100%|██████████| 556/556 [01:43<00:00,  5.39it/s]


--- Reranking Performance (BAAI/bge-reranker-large) ---
Total time for reranking 556 queries: 103.12 seconds
Average reranking latency: 185.47 ms/query
Step 3: Calculating metrics...
Saving 556 final results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/BM25___BGE-Large.jsonl...
Saved successfully.
--- Final Metrics for BM25Retriever + bge-reranker-large (k=3) ---
Average nDCG@3: 0.4469
Average Recall@3: 0.4592
------------------------------

--- Reranking: Dense_MiniLM + BGE-Large ---
--- Starting Evaluation for: DenseRetriever + bge-reranker-large ---
Step 1: Loading pre-computed initial runs for 556 queries...
Pre-computed runs loaded.
Step 2: Reranking top 30 candidates for each query using BAAI/bge-reranker-large...


Reranking (bge-reranker-large): 100%|██████████| 556/556 [01:43<00:00,  5.37it/s]


--- Reranking Performance (BAAI/bge-reranker-large) ---
Total time for reranking 556 queries: 103.54 seconds
Average reranking latency: 186.23 ms/query
Step 3: Calculating metrics...
Saving 556 final results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/Dense_MiniLM___BGE-Large.jsonl...
Saved successfully.
--- Final Metrics for DenseRetriever + bge-reranker-large (k=3) ---
Average nDCG@3: 0.9196
Average Recall@3: 0.9526
------------------------------

--- Reranking: Dense_BGE-Small + BGE-Large ---
--- Starting Evaluation for: DenseRetriever + bge-reranker-large ---
Step 1: Loading pre-computed initial runs for 556 queries...
Pre-computed runs loaded.
Step 2: Reranking top 30 candidates for each query using BAAI/bge-reranker-large...


...

--- Reranking Performance (cross-encoder/ms-marco-MiniLM-L-12-v2) ---
Total time for reranking 556 queries: 18.07 seconds
Average reranking latency: 32.51 ms/query
Step 3: Calculating metrics...
Saving 556 final results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/Dense_BGE-Small___MiniLM-L12.jsonl...
Saved successfully.
--- Final Metrics for DenseRetriever + ms-marco-MiniLM-L-12-v2 (k=3) ---
Average nDCG@3: 0.9107
Average Recall@3: 0.9496
------------------------------

--- Reranking: Hybrid_MiniLM_RRF + MiniLM-L12 ---
--- Starting Evaluation for: Hybrid_rrf_DenseRetriever + ms-marco-MiniLM-L-12-v2 ---
Step 1: Loading pre-computed initial runs for 556 queries...
Pre-computed runs loaded.
Step 2: Reranking top 30 candidates for each query using cross-encoder/ms-marco-MiniLM-L-12-v2...


Reranking (ms-marco-MiniLM-L-12-v2): 100%|██████████| 556/556 [00:17<00:00, 31.06it/s]


--- Reranking Performance (cross-encoder/ms-marco-MiniLM-L-12-v2) ---
Total time for reranking 556 queries: 17.90 seconds
Average reranking latency: 32.20 ms/query
Step 3: Calculating metrics...
Saving 556 final results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/Hybrid_MiniLM_RRF___MiniLM-L12.jsonl...
Saved successfully.
--- Final Metrics for Hybrid_rrf_DenseRetriever + ms-marco-MiniLM-L-12-v2 (k=3) ---
Average nDCG@3: 0.9092
Average Recall@3: 0.9460
------------------------------

--- Reranking: Hybrid_BGE-Small_RRF + MiniLM-L12 ---
--- Starting Evaluation for: Hybrid_rrf_DenseRetriever + ms-marco-MiniLM-L-12-v2 ---
Step 1: Loading pre-computed initial runs for 556 queries...
Pre-computed runs loaded.
Step 2: Reranking top 30 candidates for each query using cross-encoder/ms-marco-MiniLM-L-12-v2...


Reranking (ms-marco-MiniLM-L-12-v2): 100%|██████████| 556/556 [00:18<00:00, 29.75it/s]


--- Reranking Performance (cross-encoder/ms-marco-MiniLM-L-12-v2) ---
Total time for reranking 556 queries: 18.69 seconds
Average reranking latency: 33.62 ms/query
Step 3: Calculating metrics...
Saving 556 final results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/Hybrid_BGE-Small_RRF___MiniLM-L12.jsonl...
Saved successfully.
--- Final Metrics for Hybrid_rrf_DenseRetriever + ms-marco-MiniLM-L-12-v2 (k=3) ---
Average nDCG@3: 0.9112
Average Recall@3: 0.9496
------------------------------

Unloading MiniLM-L12...
Cleanup for MiniLM-L12 complete.


# Đánh giá và Kết luận

## Bảng đánh giá

---

## **Giải thích các Metric**

### **1. Recall@10 (Độ phủ)**

- Recall đo lường **khả năng tìm lại được tài liệu đúng**. Nó trả lời câu hỏi: "Trong tất cả các tài liệu đúng có thể có, model đã tìm thấy được bao nhiêu phần trăm trong top 10 kết quả?"
- **Ví dụ dễ hiểu:** Giả sử cho một query, có tổng cộng 5 tài liệu được đánh dấu là "đúng".
  - Nếu hệ thống trả về 10 kết quả, trong đó có 2 tài liệu đúng.
  - Recall@10 sẽ là `2 / 5 = 0.4` (hay 40%).
- `Recall@10 = 0.3557` của Dense là cao nhất, nghĩa là trung bình, nó tìm thấy được khoảng 35.6% số tài liệu đúng trong top 10 kết quả trả về.

### **2. nDCG@10 (Normalized Discounted Cumulative Gain)**

- Đây là metric quan trọng nhất, nó đo lường **chất lượng xếp hạng**. Nó không chỉ quan tâm mô hình có tìm được tài liệu đúng hay không, mà còn quan tâm mô hình **xếp chúng ở vị trí nào**.
- **Ví dụ dễ hiểu:** Google search. Một kết quả đúng ở vị trí #1 giá trị hơn rất nhiều so với một kết quả đúng ở vị trí #10. nDCG được thiết kế để phản ánh điều này.
  - **Gain (Lợi ích):** Model được "điểm" khi tìm thấy một tài liệu đúng.
  - **Cumulative (Tích lũy):** Điểm được cộng dồn cho tất cả các kết quả đúng trong top 10.
  - **Discounted (Chiết khấu):** Điểm của một tài liệu đúng sẽ bị "giảm giá" (chiết khấu) nếu nó nằm ở vị trí thấp. Kết quả ở vị trí #1 được 100% điểm, vị trí #2 ít hơn, và cứ thế giảm dần.
  - **Normalized (Chuẩn hóa):** Điểm số cuối cùng được chia cho điểm số của một bảng xếp hạng "hoàn hảo" lý tưởng, để đưa về thang điểm từ 0 đến 1. Điều này giúp so sánh công bằng giữa các query khác nhau.
- `nDCG@10 = 0.2411` của Dense là cao nhất, cho thấy nó không chỉ tìm được nhiều tài liệu đúng (như Recall đã chỉ ra) mà còn xếp chúng ở những vị trí rất cao trong danh sách kết quả.

---

## **Giải thích các phương pháp Hybrid**

Cả hai phương pháp đều nhằm mục đích kết hợp kết quả từ BM25 (sparse) và Dense để tạo ra một danh sách xếp hạng cuối cùng tốt hơn.

### **1. Weighted Sum Fusion (Tổng có trọng số)**

- **Cách hoạt động:** Nó hoạt động như một cái cân.
  1.  Nó lấy điểm số của BM25 và Dense.
  2.  Vì thang điểm của hai thằng này khác nhau (BM25 có thể > 20, Dense chỉ từ -1 đến 1), nó phải **chuẩn hóa (normalize)** cả hai về cùng một thang đo, thường là từ 0 đến 1.
  3.  Nó tính điểm cuối cùng bằng công thức: `Điểm cuối = (alpha * điểm_dense) + ((1 - alpha) * điểm_bm25)`.
- **Ưu điểm:** Rất đơn giản, dễ hiểu. Tham số `alpha` giống như một cái núm vặn cho phép ta điều chỉnh "độ tin tưởng" vào mỗi retriever. Nếu mày thấy Dense tốt hơn, mày tăng `alpha` lên (ví dụ 0.8).
- **Nhược điểm:** Rất nhạy cảm với cách chuẩn hóa điểm số. Nếu việc chuẩn hóa không tốt, một trong hai retriever có thể "lấn át" retriever còn lại một cách không công bằng. Kết quả (`0.2348`) khá tốt, cho thấy việc chuẩn hóa đã hoạt động tương đối ổn.

### **2. Reciprocal Rank Fusion (RRF - Hợp nhất Thứ hạng Nghịch đảo)**

- **Cách hoạt động:** Phương pháp này **hoàn toàn không quan tâm đến điểm số**, nó chỉ quan tâm đến **thứ hạng (rank)**.
  1.  Nó không nhìn vào điểm `0.9` hay `25.0`. Nó chỉ nhìn: "Tài liệu A đứng thứ **1** trong danh sách của Dense và đứng thứ **5** trong danh sách của BM25".
  2.  Nó tính điểm cho mỗi tài liệu bằng công thức: `Điểm RRF = $$\frac(1 / (k + rank_dense)) + (1 / (k + rank_bm25))$$.
  3.  Hằng số `k` (thường là 60) được thêm vào để tránh việc các thứ hạng cao có điểm số quá chênh lệch so với các thứ hạng thấp.
- **Ưu điểm:** Cực kỳ ổn định. Vì nó không dùng điểm số, nó miễn nhiễm với vấn đề thang đo khác nhau giữa BM25 và Dense. Đây thường là lựa chọn "an toàn" và hiệu quả khi ta không chắc chắn về chất lượng điểm số.
- **Nhược điểm:** Nó bỏ qua thông tin về "mức độ tự tin". Ví dụ, Dense có thể rất chắc chắn về kết quả top 1 (điểm 0.99) so với top 2 (điểm 0.7), nhưng RRF chỉ coi chúng là hạng 1 và hạng 2. Trong trường hợp này, kết quả RRF (`0.1949`) thấp hơn Weighted Sum, có thể là do việc chuẩn hóa điểm số của Weighted Sum đã hoạt động tốt một cách bất ngờ, hoặc do RRF đã làm mất đi một số thông tin hữu ích từ điểm số của Dense.

## Giải thích kết quả

BM25 cho ra kết quả rất thấp. Kết quả của BM25 tệ đến mức nó trở thành "nhiễu" cho Hybrid.

BM25 thất bại trong trường hợp này vì 2 lý do chính sau:

### 1. Bộ dữ liệu được thiết kế để ưu ái Dense Retriever

Bộ dữ liệu MS MARCO này có vẻ được thiết kế để "ưu ái" (favor) Dense Retriever hơn là BM25.
Đây là những lý do chính:

- Bản chất của câu hỏi: MS MARCO được tạo ra với các câu hỏi thường là các câu hỏi tự nhiên, ví dụ: "What is the capital of France?". Các câu hỏi này đòi hỏi sự hiểu biết về ngữ nghĩa, từ đồng nghĩa, và mối quan hệ giữa các khái niệm. Với các câu hỏi tự nhiên trong MS MARCO, người dùng diễn đạt ý của họ theo nhiều cách.

  Dense Retriever, với khả năng "nhúng" (embed) các câu hỏi và tài liệu vào một không gian ngữ nghĩa, có thể nắm bắt được sự tương đồng về ý nghĩa, ngay cả khi không có từ khóa chung.

  Trong khi đó, BM25, với khả năng chỉ dựa vào từ khóa, sẽ gặp khó khăn trong việc hiểu được ý định thực sự của người hỏi.

- Đặc điểm của các tài liệu: Các tài liệu trong MS MARCO là các đoạn văn bản ngắn (passages). Các đoạn văn này thường chứa thông tin cô đọng, tập trung vào một chủ đề cụ thể. Điều này làm cho việc "nhúng" các đoạn văn thành các vector ý nghĩa trở nên dễ dàng hơn. BM25 có thể gặp khó khăn nếu các từ khóa trong câu hỏi không xuất hiện trực tiếp trong các đoạn văn, hoặc nếu các đoạn văn quá ngắn để chứa nhiều từ khóa.

## 2. Vấn đề từ vựng không khớp (Vocabulary Mismatch Problem)

Ngay cả khi không có từ đồng nghĩa, người ta vẫn có thể dùng các từ khác nhau để mô tả cùng một thứ.

- Query: "cách sửa lỗi rò rỉ ống nước"
- Tài liệu 1: "hướng dẫn khắc phục sự cố đường ống bị chảy nước"
- Tài liệu 2: "ống nước bị rò rỉ"

BM25 sẽ cho điểm Tài liệu 2 rất cao vì nó chứa chính xác các từ khóa ["ống", "nước", "rò", "rỉ"]. Nó sẽ cho điểm Tài liệu 1 rất thấp, vì các từ ["sửa", "lỗi"] không khớp với ["khắc", "phục", "sự", "cố"]. Dù tài liệu 1 mới là câu trả lời thực sự.

Dense Retriever: nó hiểu rằng "sửa lỗi", "khắc phục sự cố", "hướng dẫn" đều nằm trong một cụm ý nghĩa về "giải quyết vấn đề". Nó sẽ cho điểm cao cho cả hai, và có thể nhận ra Tài liệu 1 có giá trị hơn.

## Giải pháp

- Improve BM25 Retriever: ta đang đánh giá con cá khả năng leo cây. Vì vậy nên thay đổi bộ dataset để nó không bị bias towards any retrieval methods. Qua đây cũng cho thấy tầm quan trọng của việc xác định bài toán (Problem statements), data analysis, từ đó xác định usecase phù hợp và lựa chọn retriever phù hợp. Ngoài ra có thể sử dụng các thư viện hỗ trợ BM25 như Pyserini, các database tích hợp sẵn service search như ElasticSearch.
- Improve Dense Retriever: có thể sử dụng Milvus làm Vector Database hỗ trợ dense retriever query tốt hơn.
- Improve Hybrid Retriever:
  - Điều chỉnh trọng số alpha: Trong Weighted Sum, thử tăng alpha lên cao hơn (ví dụ 0.8 hoặc 0.9) để cho Dense có trọng số lớn hơn và giảm ảnh hưởng của BM25.
  - Chỉ dùng BM25 như một "bộ lọc": Một kỹ thuật khác là: lấy top 100 từ Dense, sau đó dùng BM25 để xếp hạng lại (re-rank) 100 kết quả đó. Cách này giới hạn phạm vi của BM25 và không để nó làm nhiễu kết quả.

## Kết luận

Đây không phải là một "lỗi" của BM25. Nó chỉ đơn giản là một sự khác biệt về cách tiếp cận. Trong một số trường hợp (ví dụ: tìm kiếm trong các tài liệu kỹ thuật, nơi từ khóa là quan trọng), BM25 vẫn có thể hoạt động tốt. Nhưng trong trường hợp của MS MARCO, Dense Retriever có lợi thế hơn hẳn. Đối với bộ dữ liệu MS MARCO, nơi các câu hỏi rất đa dạng và tự nhiên, sự "nông cạn" của BM25 bị bộc lộ hoàn toàn, dẫn đến kết quả rất tệ.
